In [1]:
loads_path = (
    "abfss://Fleet_Logistics_Engineering@onelake.dfs.fabric.microsoft.com/"
    "Fleet_Logistics_Lakehouse.Lakehouse/Files/Landing/loads.csv"
)

df_loads_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(loads_path)
)

StatementMeta(, 5cd75f28-6243-4175-a7e1-917c917944a0, 3, Finished, Available, Finished, False)

In [2]:
display(df_loads_raw)

df_loads_raw.printSchema()

print(f"Source records: {df_loads_raw.count()}")

StatementMeta(, 5cd75f28-6243-4175-a7e1-917c917944a0, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8256d2a9-d46b-40a6-8063-95c255fd5631)

root
 |-- load_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- load_date: date (nullable = true)
 |-- load_type: string (nullable = true)
 |-- weight_lbs: integer (nullable = true)
 |-- pieces: integer (nullable = true)
 |-- revenue: double (nullable = true)
 |-- fuel_surcharge: double (nullable = true)
 |-- accessorial_charges: integer (nullable = true)
 |-- load_status: string (nullable = true)
 |-- booking_type: string (nullable = true)

Source records: 85410


In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    DateType
)

loads_path = (
    "abfss://Fleet_Logistics_Engineering@onelake.dfs.fabric.microsoft.com/"
    "Fleet_Logistics_Lakehouse.Lakehouse/Files/Landing/loads.csv"
)


StatementMeta(, 6649be54-89b7-4491-a906-306dfca83fb9, 3, Finished, Available, Finished, False)

In [2]:
load_schema = StructType([
    StructField("load_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("route_id", StringType(), True),
    StructField("load_date", DateType(), True),
    StructField("load_type", StringType(), True),
    StructField("weight_lbs", IntegerType(), True),
    StructField("pieces", IntegerType(), True),
    StructField("revenue", DoubleType(), True),
    StructField("fuel_surcharge", DoubleType(), True),
    StructField("accessorial_charges", IntegerType(), True),
    StructField("load_status", StringType(), True),
    StructField("booking_type", StringType(), True)
])

df_loads = (
    spark.read
    .option("header", "true")
    .schema(load_schema)
    .csv(loads_path)
)

display(df_loads)

print(f"Source records: {df_loads.count()}")

StatementMeta(, 6649be54-89b7-4491-a906-306dfca83fb9, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5f0b72db-bb63-45cb-a507-324ac2a3ea99)

Source records: 85410


In [3]:
null_load_ids = (
    df_loads
    .filter(F.col("load_id").isNull())
    .count()
)
print(f"NULL load IDs: {null_load_ids}")

StatementMeta(, 6649be54-89b7-4491-a906-306dfca83fb9, 5, Finished, Available, Finished, False)

NULL load IDs: 0


In [4]:
duplicate_load_ids = (
    df_loads
    .groupBy("load_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
print(f"Duplicate load IDs: {duplicate_load_ids}")

StatementMeta(, 6649be54-89b7-4491-a906-306dfca83fb9, 6, Finished, Available, Finished, False)

Duplicate load IDs: 0


In [5]:
customer_check = (
    df_loads
    .join(
        spark.table("bronze_customers").select("customer_id"),
        on="customer_id",
        how="left_anti"
    )
)

missing_customers = customer_check.count()

print(f"Loads with missing customer IDs: {missing_customers}")

StatementMeta(, 6649be54-89b7-4491-a906-306dfca83fb9, 7, Finished, Available, Finished, False)

Loads with missing customer IDs: 0


In [6]:
route_check = (
    df_loads
    .join(
        spark.table("bronze_routes").select("route_id"),
        on="route_id",
        how="left_anti"
    )
)

missing_routes = route_check.count()

print(f"Loads with missing route IDs: {missing_routes}")

StatementMeta(, 6649be54-89b7-4491-a906-306dfca83fb9, 8, Finished, Available, Finished, False)

Loads with missing route IDs: 0


In [7]:
invalid_weight = (
    df_loads
    .filter(F.col("weight_lbs") < 0)
    .count()
)
print(f"Negative weight records: {invalid_weight}")

StatementMeta(, 6649be54-89b7-4491-a906-306dfca83fb9, 9, Finished, Available, Finished, False)

Negative weight records: 0


In [8]:
invalid_pieces = (
    df_loads
    .filter(F.col("pieces") < 0)
    .count()
)
print(f"Negative pieces records: {invalid_pieces}")

StatementMeta(, 6649be54-89b7-4491-a906-306dfca83fb9, 10, Finished, Available, Finished, False)

Negative pieces records: 0


In [9]:
invalid_revenue = (
    df_loads
    .filter(F.col("revenue") < 0)
    .count()
)
print(f"Negative revenue records: {invalid_revenue}")

StatementMeta(, 6649be54-89b7-4491-a906-306dfca83fb9, 11, Finished, Available, Finished, False)

Negative revenue records: 0


In [10]:
invalid_fuel_surcharge = (
    df_loads
    .filter(F.col("fuel_surcharge") < 0)
    .count()
)
print(f"Negative fuel surcharge records: {invalid_fuel_surcharge}")

StatementMeta(, 6649be54-89b7-4491-a906-306dfca83fb9, 12, Finished, Available, Finished, False)

Negative fuel surcharge records: 0


In [11]:
invalid_accessorial = (
    df_loads
    .filter(F.col("accessorial_charges") < 0)
    .count()
)
print(f"Negative accessorial charge records: {invalid_accessorial}")

StatementMeta(, 6649be54-89b7-4491-a906-306dfca83fb9, 13, Finished, Available, Finished, False)

Negative accessorial charge records: 0


In [13]:
null_load_dates = (
    df_loads
    .filter(F.col("load_date").isNull())
    .count()
)

print(f"NULL load dates: {null_load_dates}")

StatementMeta(, 6649be54-89b7-4491-a906-306dfca83fb9, 15, Finished, Available, Finished, False)

NULL load dates: 0


In [14]:
# FAIL ETL IF DATA QUALITY RULES ARE VIOLATED
if null_load_ids > 0:
    raise ValueError("ETL failed: NULL load_id values detected.")

if duplicate_load_ids > 0:
    raise ValueError("ETL failed: Duplicate load_id values detected.")

if missing_customers > 0:
    raise ValueError(
        "ETL failed: Loads contain customer IDs not found in bronze_customers."
    )

if missing_routes > 0:
    raise ValueError(
        "ETL failed: Loads contain route IDs not found in bronze_routes."
    )

if invalid_weight > 0:
    raise ValueError("ETL failed: Negative weight values detected.")

if invalid_pieces > 0:
    raise ValueError("ETL failed: Negative pieces values detected.")

if invalid_revenue > 0:
    raise ValueError("ETL failed: Negative revenue values detected.")

if invalid_fuel_surcharge > 0:
    raise ValueError("ETL failed: Negative fuel surcharge values detected.")

if invalid_accessorial > 0:
    raise ValueError("ETL failed: Negative accessorial charges detected.")

if null_load_dates > 0:
    raise ValueError("ETL failed: NULL load_date values detected.")

print("Load data quality and referential-integrity validation passed.")

StatementMeta(, 6649be54-89b7-4491-a906-306dfca83fb9, 16, Finished, Available, Finished, False)

Load data quality and referential-integrity validation passed.


In [15]:
df_loads = (
    df_loads
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.lit("loads.csv"))
)

df_loads.createOrReplaceTempView("loads_source")

print("Load metadata added successfully.")

StatementMeta(, 6649be54-89b7-4491-a906-306dfca83fb9, 17, Finished, Available, Finished, False)

Load metadata added successfully.


In [16]:
spark.sql("""
CREATE TABLE IF NOT EXISTS bronze_loads (
    load_id STRING,
    customer_id STRING,
    route_id STRING,
    load_date DATE,
    load_type STRING,
    weight_lbs INT,
    pieces INT,
    revenue DOUBLE,
    fuel_surcharge DOUBLE,
    accessorial_charges INT,
    load_status STRING,
    booking_type STRING,
    ingestion_timestamp TIMESTAMP,
    source_file STRING
)
USING DELTA
""")

print("bronze_loads table is ready.")

StatementMeta(, 6649be54-89b7-4491-a906-306dfca83fb9, 18, Finished, Available, Finished, False)

bronze_loads table is ready.


In [17]:
result = spark.sql("""
MERGE INTO bronze_loads AS target

USING loads_source AS source

ON target.load_id = source.load_id

WHEN MATCHED THEN
    UPDATE SET
        target.customer_id = source.customer_id,
        target.route_id = source.route_id,
        target.load_date = source.load_date,
        target.load_type = source.load_type,
        target.weight_lbs = source.weight_lbs,
        target.pieces = source.pieces,
        target.revenue = source.revenue,
        target.fuel_surcharge = source.fuel_surcharge,
        target.accessorial_charges = source.accessorial_charges,
        target.load_status = source.load_status,
        target.booking_type = source.booking_type,
        target.ingestion_timestamp = source.ingestion_timestamp,
        target.source_file = source.source_file

WHEN NOT MATCHED THEN
    INSERT (
        load_id,
        customer_id,
        route_id,
        load_date,
        load_type,
        weight_lbs,
        pieces,
        revenue,
        fuel_surcharge,
        accessorial_charges,
        load_status,
        booking_type,
        ingestion_timestamp,
        source_file
    )

    VALUES (
        source.load_id,
        source.customer_id,
        source.route_id,
        source.load_date,
        source.load_type,
        source.weight_lbs,
        source.pieces,
        source.revenue,
        source.fuel_surcharge,
        source.accessorial_charges,
        source.load_status,
        source.booking_type,
        source.ingestion_timestamp,
        source.source_file
    )
""")

print("Load Bronze MERGE completed successfully.")

StatementMeta(, 6649be54-89b7-4491-a906-306dfca83fb9, 19, Finished, Available, Finished, False)

Load Bronze MERGE completed successfully.


In [18]:
bronze_count = spark.sql("""
    SELECT COUNT(*) AS record_count
    FROM bronze_loads
""").collect()[0]["record_count"]

print(f"Bronze load records: {bronze_count}")

display(
    spark.sql("""
        SELECT *
        FROM bronze_loads
        ORDER BY load_id
        LIMIT 10
    """)
)

StatementMeta(, 6649be54-89b7-4491-a906-306dfca83fb9, 20, Finished, Available, Finished, False)

Bronze load records: 85410


SynapseWidget(Synapse.DataFrame, 50e0c111-af3e-425c-8e1d-6ccf2f15d9d3)